<a href="https://colab.research.google.com/github/amarala931/MIA/blob/main/entrenamiento_supervisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Curso Práctico: Introducción al Aprendizaje Supervisado con Scikit-Learn

¡Bienvenidos! En este cuaderno aprenderemos cómo una computadora puede "aprender" a partir de datos reales utilizando uno de los algoritmos más fundamentales de la Inteligencia Artificial: la **Regresión Lineal Simple**.

En esta ocasión, utilizaremos un dataset real integrado en la librería **scikit-learn** para predecir la **progresión de la diabetes** en un paciente a partir de su **Índice de Masa Corporal (IMC o BMI)**. ¡Un enfoque 100% práctico y aplicado a la salud!

# Caso de Estudio: Predicción Médica en Pacientes con Diabetes

Analizaremos cómo el Índice de Masa Corporal de una persona se relaciona directamente con el nivel de progresión de la enfermedad al cabo de un año. De este modo, entrenaremos una inteligencia artificial para estimar el avance médico basándonos en indicadores físicos del paciente.

## Sección 1: ¿Qué es el aprendizaje supervisado?

Imagina que queremos que una computadora nos ayude a estimar la evolución médica de un paciente de forma automatizada. Para ello, le enseñaremos utilizando **Aprendizaje Supervisado**.

En el contexto de la salud, esto funciona así:
1. Le damos al algoritmo un conjunto de datos históricos con respuestas reales que ya conocemos. A esto lo llamamos **etiquetas** o **variable objetivo ($y$)**. En nuestro caso, la progresión anual de la diabetes.
2. También le proporcionamos las condiciones biológicas conocidas como **características** o **variables predictoras ($X$)**. En nuestro caso, el Índice de Masa Corporal (IMC).

El objetivo del algoritmo de Regresión Lineal es trazar el camino óptimo entre el IMC ($X$) y la progresión médica ($y$) para poder realizar estimaciones fiables sobre nuevos pacientes en el futuro.

## Sección 2: Obtención y exploración de datos reales de Scikit-Learn

Cargaremos el dataset de diabetes de `scikit-learn`, extraeremos la característica del Índice de Masa Corporal (BMI) como nuestra variable predictora ($X$) y la medida cuantitativa de la progresión de la enfermedad como variable objetivo ($y$).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# 1. Cargamos el dataset oficial
diabetes = load_diabetes(as_frame=True)
df_completo = diabetes.frame

# Para nuestra regresión simple, seleccionaremos:
# X: 'bmi' (Índice de masa corporal)
# y: 'target' (Progresión de la enfermedad al año)
df = df_completo[['bmi', 'target']].copy()
df.columns = ['BMI', 'Progreso_Enfermedad']

# Tomamos una muestra representativa de 40 pacientes para que el gráfico sea muy claro
df = df.sample(n=40, random_state=42).reset_index(drop=True)

print("=== DATOS OBTENIDOS DE SCIKIT-LEARN (Muestra de pacientes) ===")
display(df.head(10))

# Imprimimos las dimensiones del dataset
filas, columnas = df.shape
print(f"\nDimensiones del Dataset seleccionado: {filas} pacientes y {columnas} variables de interés.")

### Visualización Exploratoria Inicial

Antes de entrenar un modelo, es vital que los alumnos vean si existe de forma visual alguna relación o tendencia entre ambas variables mediante un gráfico de dispersión.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='BMI', y='Progreso_Enfermedad', color='purple', s=70)
plt.title('Análisis Exploratorio: IMC vs Progresión de la Diabetes')
plt.xlabel('Índice de Masa Corporal (Normalizado)')
plt.ylabel('Progresión de la Enfermedad')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Sección 3: División en Entrenamiento y Prueba (Train/Test Split)

Si estudias para un examen memorizando exactamente las preguntas del libro, ¿realmente aprendiste o solo memorizaste? Para saber si de verdad aprendiste, el profesor debe evaluarte con preguntas nuevas.

En Ciencia de Datos hacemos lo mismo: **dividimos nuestros datos en dos grupos**:
1. **Conjunto de Entrenamiento (Train):** Los datos con los que el modelo "estudia" y aprende la relación (usaremos el 80% de los datos).
2. **Conjunto de Prueba (Test):** Datos que el modelo no verá durante su aprendizaje. Los usaremos al final para "evaluar" qué tan bueno es prediciendo (usaremos el 20% restante).

In [ ]:
from sklearn.model_selection import train_test_split

# Definimos X e y
X = df[['BMI']]
y = df['Progreso_Enfermedad']

# Dividimos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("=== CONJUNTO DE ENTRENAMIENTO (Estudio) ===")
train_data = X_train.copy()
train_data['Progreso_Enfermedad'] = y_train
display(train_data.head(5))

print("\n=== CONJUNTO DE PRUEBA (Evaluación) ===")
test_data = X_test.copy()
test_data['Progreso_Enfermedad'] = y_test
display(test_data.head(5))

print(f"\nResumen: Entrenamos con {len(X_train)} pacientes y evaluamos con {len(X_test)} pacientes.")

### Visualización del Split

Mostremos gráficamente a los alumnos cómo se distribuyeron aleatoriamente los pacientes en el conjunto de entrenamiento y en el de prueba para que comprendan de forma visual el concepto de partición.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Entrenamiento (80%)', alpha=0.7, s=60)
plt.scatter(X_test, y_test, color='orange', label='Prueba (20%)', alpha=0.9, s=80, marker='D')
plt.title('Distribución Visual del Train/Test Split')
plt.xlabel('Índice de Masa Corporal (BMI)')
plt.ylabel('Progresión de la Enfermedad')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Sección 4: Entrenamiento del modelo

La **Regresión Lineal Simple** asume que existe una tendencia directa y lineal entre el indicador físico de nuestro paciente y su estado de salud futuro. La ecuación de la línea que nuestro modelo ajustará es:

$$\text{Progresión} = (m \times \text{IMC}) + b$$

Donde:
*   **$m$ (Pendiente o Coeficiente):** Indica cuánto se acelera o incrementa la progresión de la diabetes por cada incremento unitario en el Índice de Masa Corporal.
*   **$b$ (Intercepto o Sesgo):** Es la tasa base estimada de progresión de la enfermedad para un valor promedio o neutral del indicador físico.

¡Vamos a entrenar el modelo en Scikit-Learn para que encuentre de forma óptima los coeficientes de esta ecuación!

In [ ]:
from sklearn.linear_model import LinearRegression

# Instanciar y entrenar
modelo_diabetes = LinearRegression()
modelo_diabetes.fit(X_train, y_train)

pendiente_db = modelo_diabetes.coef_[0]
intercepto_db = modelo_diabetes.intercept_

print("=== PARÁMETROS APRENDIDOS POR EL MODELO ===")
print(f"Pendiente (m): {pendiente_db:.4f} (Indica el impacto del incremento del BMI)")
print(f"Intercepto (b): {intercepto_db:.4f} (Punto de partida del progreso médico)")
print("\n=== ECUACIÓN MATEMÁTICA DETECTADA ===")
print(f"Progreso_Enfermedad = {pendiente_db:.4f} * (BMI) + {intercepto_db:.4f}")

## Sección 5: Predicción y cálculo del error

Con nuestra ecuación clínica ya calibrada por el algoritmo, utilizaremos el conjunto de prueba para generar predicciones y medir la precisión de nuestro modelo.

Calcularemos de manera transparente la diferencia entre el valor médico real y la predicción matemática del modelo (el error absoluto) junto con el Error Absoluto Medio (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error

# Predicciones
predicciones_db = modelo_diabetes.predict(X_test)

# Tabla comparativa
tabla_comparativa_db = pd.DataFrame({
    'BMI del Paciente': X_test['BMI'],
    'Progreso Real': y_test,
    'Progreso Predicho': predicciones_db
})
tabla_comparativa_db['Error Absoluto'] = (tabla_comparativa_db['Progreso Real'] - tabla_comparativa_db['Progreso Predicho']).abs()

print("=== RESULTADOS DE PREDICCIÓN ===")
display(tabla_comparativa_db.round(2))

# MAE
mae_db = mean_absolute_error(y_test, predicciones_db)
print(f"\nError Absoluto Medio (MAE): {mae_db:.2f} unidades de progresión.")

## Sección 6: Visualización del aprendizaje de la Regresión Lineal

Para cerrar la clase de forma interactiva, graficaremos todos nuestros puntos de entrenamiento, los puntos de evaluación y la línea de regresión continua calculada por nuestro modelo. Esto permitirá que los alumnos vean claramente cómo el modelo traza la predicción óptima reduciendo la distancia (los errores) en cada punto evaluado.

In [ ]:
plt.figure(figsize=(10, 6))

# Puntos reales de entrenamiento
plt.scatter(X_train, y_train, color='blue', label='Pacientes de Entrenamiento', s=70, alpha=0.6)

# Puntos reales de prueba
plt.scatter(X_test, y_test, color='orange', label='Pacientes de Prueba (Reales)', s=90, marker='D', alpha=0.8)

# Línea del modelo de regresión
x_rango = np.linspace(df['BMI'].min() - 0.02, df['BMI'].max() + 0.02, 100).reshape(-1, 1)
y_rango_pred = modelo_diabetes.predict(x_rango)

plt.plot(x_rango, y_rango_pred, color='red', linewidth=3, label='Línea de Ajuste del Modelo')

# Trazar líneas de error para las predicciones de prueba
for x_val, y_real, y_pred in zip(X_test['BMI'], y_test, predicciones_db):
    plt.plot([x_val, x_val], [y_real, y_pred], color='gray', linestyle=':', alpha=0.8)

plt.title('Visualización Final: Modelo de Regresión Lineal con Dataset de Diabetes', fontsize=12, fontweight='bold')
plt.xlabel('BMI (Índice de Masa Corporal Normalizado)')
plt.ylabel('Progresión de la Enfermedad')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

## Sección 7: Predicción para nuevos pacientes clínicos

¡Es hora de simular el uso de nuestro modelo en una clínica u hospital real! Crearemos datos de pacientes completamente nuevos con diferentes niveles de IMC y dejaremos que nuestro modelo entrenado calcule la progresión estimada para cada uno.

In [ ]:
# 1. Definimos nuevos valores de IMC (algunos bajos, promedios y elevados)
# Nota: Los datos en scikit-learn ya están normalizados entre aprox -0.1 y 0.17
nuevos_pacientes_bmi = np.array([-0.05, 0.0, 0.04, 0.12]).reshape(-1, 1)

# 2. El modelo realiza la predicción
predicciones_nuevas = modelo_diabetes.predict(nuevos_pacientes_bmi)

# 3. Creamos una tabla explicativa para mostrar a los alumnos
clinica_df = pd.DataFrame({
    'Paciente': ['Paciente A (IMC Bajo)', 'Paciente B (IMC Promedio)', 'Paciente C (IMC Moderado)', 'Paciente D (IMC Alto)'],
    'IMC Registrado': nuevos_pacientes_bmi.flatten(),
    'Progresión Estimada de Diabetes': predicciones_nuevas
})

print("=== PREDICCIONES GENERADAS PARA NUEVOS CASOS EN LA CLÍNICA ===")
display(clinica_df.round(2))

# Explicación del funcionamiento de la fórmula matemática para el Paciente D
paciente_d_imc = nuevos_pacientes_bmi[3][0]
prediccion_d_manual = (pendiente_db * paciente_d_imc) + intercepto_db
print(f"\nDemostración matemática del Paciente D:")
print(f"Fórmula: Progreso = ({pendiente_db:.2f} * {paciente_d_imc:.2f}) + {intercepto_db:.2f}")
print(f"Resultado calculado a mano: {prediccion_d_manual:.2f} (Coincide con el modelo)")